# 10. Memory Profiling & In-Place Optimization (5+ Years Interview Guide)
Exhaustive revision guide to temporary array elimination, out= pre-allocated buffers, in-place operators, and memory profiling on transaction vectors.

### Key 5-Year Interview Concepts Covered:
- **In-Place Parameters (`out=`)**: Directing results into pre-allocated memory buffers (`np.add(a, b, out=a)`).
- **In-Place Operators (`+=`, `*=`)**: Mutating memory in-place vs creating temporary arrays.
- **Memory Profiling (`sys.getsizeof` & `.nbytes`)**: Isolating memory consumption.
- **Zero-Allocation Pipelines**: Chaining mathematical operations without intermediate heap allocations.

This interactive revision guide loads and operates directly on `data/raw_transactions.csv` using dedicated cells per method.

In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

NumPy Version: 1.26.4
Loaded from data/raw_transactions.csv (14262 clean aligned rows):
- amounts array: shape (14262,), dtype float64
- fraud_flags array: shape (14262,), dtype int8
- account_ages array: shape (14262,), dtype float32


### In-Place Mutation with `+=` and `*=`
**Explanation**: Scales transaction amounts in-place without heap allocations.

**Syntax**: `amt_buf *= 1.02`

In [2]:
amt_buf = amounts[:1000].copy()
orig_id = id(amt_buf)
amt_buf *= 1.05  # In-place
print('Memory address preserved after in-place mutation?:', id(amt_buf) == orig_id)

Memory address preserved after in-place mutation?: True


### Pre-Allocated Buffers with `out=`
**Explanation**: Directs addition of fee buffers into a pre-allocated destination.

**Syntax**: `np.add(amounts[:1000], 5.0, out=dest_buffer)`

In [3]:
dest_buffer = np.empty_like(amounts[:1000])
np.add(amounts[:1000], 2.50, out=dest_buffer)
print('Output Buffer Result (first 5):', dest_buffer[:5].round(2))

Output Buffer Result (first 5): [1218.83  327.49  139.16  126.71 1287.18]


### Memory Profiling: `sys.getsizeof()` vs `.nbytes`
**Explanation**: Compares raw binary data memory vs Python wrapper struct overhead.

**Syntax**: `amounts.nbytes` vs `sys.getsizeof(amounts)`

In [4]:
print(f'Raw Binary Buffer: {amounts.nbytes / 1024:.2f} KB')
print(f'Wrapper Object Overhead: {sys.getsizeof(amounts)} bytes')

Raw Binary Buffer: 111.42 KB
Wrapper Object Overhead: 112 bytes


## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: Zero-Allocation Fee & Tax Formula Pipeline
**Explanation**: Compute `final_amount = (amount * 1.02) + 0.30` without allocating any intermediate temporary arrays.

**Syntax**: `np.multiply(amounts, 1.02, out=res); np.add(res, 0.30, out=res)`

In [5]:
res = np.empty(1000, dtype=np.float64)
np.multiply(amounts[:1000], 1.02, out=res)
np.add(res, 0.30, out=res)
print('Zero-Allocation Pipeline Result Head:', res[:5].round(2))

Zero-Allocation Pipeline Result Head: [1240.96  331.79  139.69  126.99 1310.67]
